# **Aula 4**: Inferência Estatística

OBJETIVOS:

 - Avaliar se diferenças observadas nos dados são estatisticamente significativas.

Usaremos o Data Warehouse construído na Aula 03: "dw_olist.csv"

In [0]:
import pandas as pd
import numpy as np
import time
from scipy import stats

In [0]:
# Leitura do Data Warehouse (Star Schema)
#
# Esse arquivo foi gerado ao final da Aula 03

dw = pd.read_csv("dw_olist.csv")

dw.head()

# Perguntas:

Clientes de estados diferentes gastam, em média, valores diferentes?

Antes:
- Apenas observávamos médias

Agora:
- Vamos testar se a diferença é estatisticamente real

## Seleção de dois grupos para comparação

Exemplo:
- Clientes de SP
- Clientes de RJ

Poderíamos trocar por qualquer outro critério

In [0]:
sp = dw[dw['customer_state'] == 'SP']['price']
rj = dw[dw['customer_state'] == 'RJ']['price']

print("Tamanho da amostra SP:", sp.shape[0])
print("Tamanho da amostra RJ:", rj.shape[0])

## Estatística descritiva básica

Antes de qualquer teste estatístico, sempre analisamos médias e dispersão


In [0]:
print("Média SP:", sp.mean())
print("Média RJ:", rj.mean())

print("\nDesvio padrão SP:", sp.std())
print("Desvio padrão RJ:", rj.std())

## HIPÓTESES ESTATÍSTICAS

H0 (Hipótese Nula):
- Não há diferença entre as médias

H1 (Hipótese Alternativa):
- As médias são diferentes

Nível de significância:
- alpha = 0.05

In [0]:
alpha = 0.05

## Teste de normalidade (Shapiro-Wilk)

Muitos testes paramétricos assumem normalidade

Hipóteses do teste:
- H0: Os dados seguem uma distribuição normal
- H1: Os dados NÃO seguem uma distribuição normal

Regra de decisão:
- Se p-valor > 0.05 → NÃO rejeitamos H0 (normalidade plausível)
- Se p-valor <= 0.05 → Rejeitamos H0 (não normal)

In [0]:
amostra_sp = sp.sample(500, random_state=42)
amostra_rj = rj.sample(500, random_state=42)

# Teste Shapiro-Wilk
stat_sp, p_sp = stats.shapiro(amostra_sp)
stat_rj, p_rj = stats.shapiro(amostra_rj)

print("Shapiro-Wilk - SP")
print("Estatística:", stat_sp)
print("p-valor:", p_sp)

if p_sp > 0.05:
    print("Conclusão: Não rejeitamos H0 → distribuição aproximadamente normal\n")
else:
    print("Conclusão: Rejeitamos H0 → distribuição não normal\n")

print("Shapiro-Wilk - RJ")
print("Estatística:", stat_rj)
print("p-valor:", p_rj)

if p_rj > 0.05:
    print("Conclusão: Não rejeitamos H0 → distribuição aproximadamente normal")
else:
    print("Conclusão: Rejeitamos H0 → distribuição não normal")

## JUSTIFICATIVA PARA O USO DO TESTE t

Apesar de o teste de Shapiro-Wilk indicar que as distribuições originais não são normais, podemos utilizar o Teste t devido ao:

## TEOREMA CENTRAL DO LIMITE (TCL)

Com amostras grandes:
- A distribuição da média tende à normal
- O Teste t torna-se robusto à não normalidade

Neste caso:
- n(SP) e n(RJ) são grandes
- O uso do Teste t é estatisticamente válido

In [0]:
print("Tamanho da amostra SP:", sp.shape[0])
print("Tamanho da amostra RJ:", rj.shape[0])

## Teste t para duas amostras independentes

equal_var=False -> variâncias possivelmente diferentes

In [0]:
t_stat, p_valor = stats.ttest_ind(sp, rj, equal_var=False)

print("Estatística t:", t_stat)
print("p-valor:", p_valor)

## INTERPRETAÇÃO

Regra de decisão:
- Se p-valor < alpha → rejeitamos H0
- Caso contrário → não rejeitamos H0

In [0]:
if p_valor < alpha:
    print("Resultado: Diferença estatisticamente significativa.")
else:
    print("Resultado: Não há evidência estatística de diferença.")

## Comparação de médias: MG vs SP

In [0]:
mg = dw[dw['customer_state'] == 'MG']['price']
sp = dw[dw['customer_state'] == 'SP']['price']

print("Média MG:", mg.mean())
print("Média SP:", sp.mean())

t_stat, p_valor = stats.ttest_ind(mg, sp, equal_var=False)

print("p-valor:", p_valor)

if p_valor < alpha:
    print("Diferença estatisticamente significativa.")
else:
    print("Não há evidência estatística de diferença.")

### Comparação de médias usando outra variável: valor do frete (freight_value)

In [0]:
frete_sp = dw[dw['customer_state'] == 'SP']['freight_value']
frete_rj = dw[dw['customer_state'] == 'RJ']['freight_value']

print("Média frete SP:", frete_sp.mean())
print("Média frete RJ:", frete_rj.mean())

t_stat, p_valor = stats.ttest_ind(frete_sp, frete_rj, equal_var=False)

print("p-valor:", p_valor)

if p_valor < alpha:
    print("Diferença estatisticamente significativa no frete.")
else:
    print("Não há evidência estatística de diferença no frete.")

## Comparação de médias: produtos grandes vs pequenos

Usando volume do produto

In [0]:
mediana_volume = dw['product_volume_cm3'].median()

produtos_grandes = dw[
    dw['product_volume_cm3'] >= mediana_volume
]['price']

produtos_pequenos = dw[
    dw['product_volume_cm3'] < mediana_volume
]['price']

print("Média (produtos grandes):", produtos_grandes.mean())
print("Média (produtos pequenos):", produtos_pequenos.mean())

t_stat, p_valor = stats.ttest_ind(
    produtos_grandes,
    produtos_pequenos,
    equal_var=False
)

print("p-valor:", p_valor)

if p_valor < alpha:
    print("Diferença estatisticamente significativa.")
else:
    print("Não há evidência estatística de diferença.")

## Antes de testar, sempre OLHAR os dados

In [0]:
dw.groupby('fim_de_semana')['price'].agg(
    ['mean', 'median', 'std', 'count']
)

## TESTE t PARA UMA AMOSTRA

Pergunta:
- O preço médio das compras é diferente de R$ 100?

In [0]:
valor_referencia = 100
precos = dw['price']

print("Média observada:", precos.mean())
print("Valor de referência:", valor_referencia)

t_stat, p_valor = stats.ttest_1samp(precos, valor_referencia)

print("Estatística t:", t_stat)
print("p-valor:", p_valor)

if p_valor < alpha:
    print("Conclusão: A média é estatisticamente diferente de R$ 100.")
else:
    print("Conclusão: Não há evidência de diferença em relação a R$ 100.")

## TESTE t UNICAUDAL (UMA AMOSTRA)

Pergunta:
- O frete médio é MAIOR que R$ 20?

In [0]:
valor_referencia = 20
frete = dw['freight_value']

print("Média do frete:", frete.mean())

t_stat, p_valor = stats.ttest_1samp(frete, valor_referencia)

# Ajuste para teste unicaudal
p_valor_unicaudal = p_valor / 2

print("p-valor (unicaudal):", p_valor_unicaudal)

if (t_stat > 0) and (p_valor_unicaudal < alpha):
    print("Conclusão: O frete médio é maior que R$ 20.")
else:
    print("Conclusão: Não há evidência de que o frete médio seja maior que R$ 20.")

## TESTE t PARA UMA AMOSTRA (SUBGRUPO)

Pergunta:
- O preço médio em SP é diferente da média nacional?

In [0]:
media_nacional = dw['price'].mean()
precos_sp = dw[dw['customer_state'] == 'SP']['price']

print("Média SP:", precos_sp.mean())
print("Média nacional:", media_nacional)

t_stat, p_valor = stats.ttest_1samp(precos_sp, media_nacional)

print("p-valor:", p_valor)

if p_valor < alpha:
    print("Conclusão: SP difere da média nacional.")
else:
    print("Conclusão: SP não difere da média nacional.")